# ARIMA Grid Search

## Purpose
Conducts an exhaustive grid search over ARIMA `(p, d, q)` parameter combinations to identify the configuration that minimises walk-forward RMSE on the training dataset.

A critical convergence guard is applied after every model fit: statsmodels emits a `ConvergenceWarning` rather than raising an exception when L-BFGS-B optimisation fails, so non-converged fits are **not** caught by a bare `except` block. The `mle_retvals['converged']` flag is checked explicitly and any non-converged fit raises a `RuntimeError`, causing that configuration to be skipped cleanly.

## Inputs
- `data/dataset.csv` — Training dataset (93 monthly observations)

## Outputs
- The RMSE for each converged configuration and the overall best configuration.

In [1]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
from math import sqrt
import numpy as np

import warnings
warnings.filterwarnings("ignore")

## Differencing and Inverse-Difference Helpers

Same seasonal differencing and inversion utilities used throughout this project. Seasonal differencing (lag-12) is applied to the history window before each ARIMA fit, and the inverse transformation is applied to bring the forecast back to the original scale.

In [2]:
# create a differenced series
def difference(dataset, interval=1):
    diff = list()
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return diff

In [3]:
# invert differenced value
def inverse_difference(history, yhat, interval=1):
    return yhat + history[-interval]

## Single-Order Evaluation Function

Runs walk-forward validation for a given `(p, d, q)` order and returns the RMSE.

**Convergence guard:** after every `model.fit()`, `model_fit.mle_retvals['converged']` is checked explicitly. If `False`, a `RuntimeError` is raised immediately. This is necessary because statsmodels signals convergence failure via a warning, not an exception — a bare `except: continue` in the outer loop would silently record the RMSE of a non-converged fit as valid, producing misleadingly low scores.

In [4]:
# evaluate an ARIMA model for a given order (p,d,q) and return RMSE
def evaluate_arima_model(X, arima_order):
    X = X.astype('float64')
    train_size = int(len(X) * 0.50)
    train, test = X[0:train_size], X[train_size:]
    history = [x for x in train]
    predictions = list()
    for t in range(len(test)):
        months_in_year = 12
        diff = difference(history, months_in_year)
        model = ARIMA(diff, order=arima_order)
        model_fit = model.fit()
        # Guard: reject fits where MLE optimisation failed to converge
        if not model_fit.mle_retvals['converged']:
            raise RuntimeError(f"ARIMA{arima_order} did not converge")
        yhat = model_fit.forecast()[0]
        yhat = inverse_difference(history, yhat, months_in_year)
        predictions.append(yhat)
        history.append(test[t])
    rmse = sqrt(mean_squared_error(test, predictions))
    return rmse

## Grid Search Over All (p, d, q) Combinations

Iterates over every combination of the supplied `p`, `d`, and `q` ranges, calls `evaluate_arima_model` for each, and tracks the configuration with the lowest RMSE. Configurations that raise any exception (non-convergence, singular matrix, etc.) are skipped via `except: continue`.

In [5]:
# evaluate combinations of p, d and q values for an ARIMA model
def evaluate_models(dataset, p_values, d_values, q_values):
    dataset = dataset.astype('float64')
    best_score, best_cfg = float("inf"), None
    for p in p_values:
        for d in d_values:
            for q in q_values:
                order = (p,d,q)
                try:
                    rmse = evaluate_arima_model(dataset, order)
                    if rmse < best_score:
                        best_score, best_cfg = rmse, order
                    print('ARIMA%s RMSE=%.3f' % (order,rmse))
                except:
                    continue
    print('Best ARIMA%s RMSE=%.3f' % (best_cfg, best_score))

## Load Training Data

In [6]:
series = pd.read_csv('data/dataset.csv', index_col=0, parse_dates=True).iloc[:, 0]
series.head()

Month
1964-01-01    2815
1964-02-01    2672
1964-03-01    2755
1964-04-01    2721
1964-05-01    2946
Name: Sales, dtype: int64

## Execute Grid Search

Search over `p ∈ [0, 6]`, `d ∈ [0, 2]`, `q ∈ [0, 6]` — 147 candidate configurations. Runtime will be several minutes due to the walk-forward re-fitting at every step for every configuration. Only configurations where every walk-forward step converges will be reported.

In [7]:
# evaluate parameters
p_values = range(0, 7)
d_values = range(0, 3)
q_values = range(0, 7)
evaluate_models(series.values, p_values, d_values, q_values)

ARIMA(0, 0, 0) RMSE=947.677
ARIMA(0, 0, 1) RMSE=948.007
ARIMA(0, 0, 2) RMSE=973.907
ARIMA(0, 0, 3) RMSE=987.019
ARIMA(0, 1, 0) RMSE=1145.923
ARIMA(0, 1, 1) RMSE=959.117
ARIMA(0, 1, 2) RMSE=958.600
ARIMA(0, 1, 3) RMSE=976.966
ARIMA(0, 2, 0) RMSE=1930.801
ARIMA(0, 2, 1) RMSE=1157.260
ARIMA(0, 2, 2) RMSE=961.481
ARIMA(1, 0, 0) RMSE=945.107
ARIMA(1, 1, 0) RMSE=1071.385
ARIMA(1, 1, 1) RMSE=961.619
ARIMA(1, 2, 0) RMSE=1560.305
ARIMA(1, 2, 1) RMSE=1081.849
ARIMA(2, 0, 0) RMSE=962.750
ARIMA(2, 1, 0) RMSE=1031.490
ARIMA(2, 1, 1) RMSE=971.916
ARIMA(2, 2, 0) RMSE=1352.151
ARIMA(2, 2, 1) RMSE=1041.759
ARIMA(3, 0, 0) RMSE=974.493
ARIMA(3, 1, 0) RMSE=1029.038
ARIMA(3, 2, 0) RMSE=1248.314
ARIMA(4, 0, 0) RMSE=996.877
ARIMA(4, 1, 0) RMSE=1047.517
ARIMA(4, 2, 0) RMSE=1231.917
ARIMA(5, 0, 0) RMSE=1015.178
ARIMA(5, 1, 0) RMSE=1043.531
ARIMA(5, 2, 0) RMSE=1243.241
ARIMA(6, 0, 0) RMSE=1024.865
ARIMA(6, 1, 0) RMSE=1018.743
ARIMA(6, 2, 0) RMSE=1225.748
Best ARIMA(1, 0, 0) RMSE=945.107
